![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# M3L2 E07 - OpenAIEmbeddings: convertir texto en vectores (Resolution)

## BLOQUE 1 — ¿Qué es un embedding?

Un **embedding** es una representación numérica del **significado** de un texto.

```text
Texto: "vacaciones"
  |
  v
Embedding: [0.012, -0.045, 0.089, ..., 0.098]  <- 1536 números
```

Cada número representa una "dimensión" del significado. Textos con significado similar tienen vectores **cercanos** en el espacio. Textos diferentes tienen vectores **lejanos**.

### Analogía: el mapa de significados

Imaginá un mapa donde:
- "vacaciones" está cerca de "licencia" y "días libres"
- "vacaciones" está lejos de "fútbol" y "programación"

Los embeddings crean ese mapa en 1536 dimensiones. No podemos visualizarlo, pero la matemática de la similitud del coseno sí puede medir distancias.

### ¿Para qué sirven en LangChain/RAG?

```text
Pregunta: "Cuantos dias de vacaciones tengo?"
  |
  v
Embedding de la pregunta: [0.12, -0.45, ...]
  |
  v
Buscar vectores SIMILARES en la base de datos (FAISS, Chroma)
  |
  v
Traer los K documentos con vectores más cercanos
  |
  v
Solo esos K documentos van al LLM como contexto
```

Sin embeddings habría que mandar **todos** los documentos al LLM cada vez. Con embeddings, solo los relevantes.

## BLOQUE 2 — Tipos de modelos de embeddings

LangChain soporta múltiples proveedores de embeddings con la misma interfaz:

| Proveedor | Clase | Dimensiones | Modelo por defecto |
|---|---|---|---|
| OpenAI | `OpenAIEmbeddings` | 1536 | `text-embedding-ada-002` |
| OpenAI (nuevo) | `OpenAIEmbeddings` | 1536 | `text-embedding-3-small` |
| HuggingFace | `HuggingFaceEmbeddings` | Varía | `sentence-transformers/all-MiniLM-L6-v2` |
| Google | `GoogleGenerativeAIEmbeddings` | 768 | `embedding-001` |
| Ollama (local) | `OllamaEmbeddings` | Varía | `nomic-embed-text` |

**Importante**: para comparar embeddings, ambos textos deben usar el **mismo** modelo de embeddings.

## BLOQUE 3 — Setup

In [ ]:
import os, getpass, math
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from langchain_openai import OpenAIEmbeddings

## BLOQUE 4 — TODO 1: Crear el modelo de embeddings

`OpenAIEmbeddings` tiene dos métodos principales:

| Método | Entrada | Salida | ¿Cuándo usarlo? |
|---|---|---|---|
| `embed_query(texto)` | 1 string | 1 vector (`List[float]`) | Cuando embedés una pregunta o consulta |
| `embed_documents(textos)` | Lista de strings | Lista de vectores (`List[List[float]]`) | Cuando embedés documentos completos |

Parámetros opcionales:
- `model`: `"text-embedding-ada-002"` (default), `"text-embedding-3-small"`, `"text-embedding-3-large"`
- `dimensions`: solo para modelos `text-embedding-3-*` (default: 1536)

In [ ]:
# TODO 1: crear el objeto OpenAIEmbeddings
embeddings = OpenAIEmbeddings()
print(f"Tipo: {type(embeddings).__name__}")
print(f"Modelo: {embeddings.model}")

## BLOQUE 5 — TODO 2: Embeder un texto y ver su estructura

El vector de salida es una lista de 1536 números de punto flotante. Cada texto produce exactamente el mismo largo de vector.

In [ ]:
# TODO 2: embeder "vacaciones"
vector = embeddings.embed_query("vacaciones")

print(f"Tipo del resultado: {type(vector).__name__}")
print(f"Dimensiones del vector: {len(vector)}")
print(f"Primeros 5 valores: {[round(x, 4) for x in vector[:5]]}")
print(f"Ultimos 5 valores:  {[round(x, 4) for x in vector[-5:]]}")
print()
print("Cada uno de estos 1536 numeros representa una dimension del significado.")
print("No son interpretables individualmente, pero el conjunto captura el significado.")

## BLOQUE 6 — La similitud del coseno

Para comparar dos vectores usamos la **similitud del coseno** :

```text
cosine_similarity(v1, v2) = (v1 · v2) / (||v1|| * ||v2||)
                           producto   /  producto de
                           punto          normas
```

Devuelve un valor entre -1 y 1:

| Valor | Significado | Ejemplo |
|---|---|---|
| ~1.0 | Muy similares | "vacaciones" vs "licencia" |
| ~0.8 | Relacionados | "vacaciones" vs "beneficios laborales" |
| ~0.3 | Poco relacionados | "vacaciones" vs "computadora" |
| ~0.0 | Sin relación | Textos aleatorios |
| Negativo | Opuestos (raro en embeddings de texto) | "feliz" vs "triste" |

In [ ]:
def cosine_similarity(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    n1 = math.sqrt(sum(a * a for a in v1))
    n2 = math.sqrt(sum(b * b for b in v2))
    return dot / (n1 * n2) if n1 and n2 else 0.0

print("Funcion cosine_similarity implementada.")
print()
# Verificar: el mismo texto debe tener similitud ~1.0
v_identico = embeddings.embed_query("vacaciones")
print(f"Mismo texto 'vacaciones' vs 'vacaciones': {cosine_similarity(vector, v_identico):.10f}")

## BLOQUE 7 — TODO 3: Comparar textos similares vs diferentes

Ahora vamos a embeder 4 textos y medir similitudes entre pares.

**Hipótesis:**
- Vacaciones vs Licencia → similitud ALTA (temática laboral/beneficios)
- Vacaciones vs Fútbol → similitud BAJA (temas distintos)
- Vacaciones vs Beneficios → similitud MEDIA-ALTA (relacionados)
- Licencia vs Beneficios → similitud ALTA (misma temática)

In [ ]:
# TODO 3
textos = [
    "Cuantos dias de vacaciones tengo?",
    "Politica de licencia y dias libres de la empresa",
    "El partido de futbol fue emocionante",
    "Quiero saber sobre los beneficios laborales",
]

vectores = embeddings.embed_documents(textos)

pares = [
    (0, 1, "Vacaciones vs Licencia (similar)"),
    (0, 2, "Vacaciones vs Futbol (diferente)"),
    (0, 3, "Vacaciones vs Beneficios (relacionado)"),
    (1, 3, "Licencia vs Beneficios (relacionado)"),
]
print("Similitud del coseno entre textos:")
for i, j, label in pares:
    sim = cosine_similarity(vectores[i], vectores[j])
    print(f"  {label}: {sim:.4f}")

## BLOQUE 8 — Ejemplos adicionales

### 8.1 Comparación con textos en inglés (cross-language)

In [ ]:
textos_en = [
    "How many vacation days do I have?",
    "Company policy on leaves and holidays",
    "The football match was exciting",
]

vectores_en = embeddings.embed_documents(textos_en)

print("Comparacion cross-language (ES vs EN, mismo significado):")
print(f"  'Vacaciones' vs 'Vacation days': {cosine_similarity(vectores[0], vectores_en[0]):.4f}")
print(f"  'Vacaciones' vs 'Football match': {cosine_similarity(vectores[0], vectores_en[2]):.4f}")
print()
print("Los embeddings capturan significado, no idioma.")

### 8.2 Visualizar la magnitud del vector (para entender la normalización)

In [ ]:
def vector_magnitude(v):
    return math.sqrt(sum(x * x for x in v))

for i, texto in enumerate(textos):
    mag = vector_magnitude(vectores[i])
    print(f"  Magnitud del vector '{texto[:40]}...': {mag:.4f}")
print()
print("Las magnitudes son similares porque los embeddings estan normalizados.")

## BLOQUE 9 — Debugging de embeddings

Cosas útiles para debuguear embeddings:

In [ ]:
print("========== DEBUG EMBEDDINGS ==========")
print(f"Modelo de embeddings: {embeddings.model}")
print(f"Dimensiones del vector: {len(vector)}")
print()

# Verificar que el mismo texto siempre da el mismo vector
v1 = embeddings.embed_query("texto de prueba")
v2 = embeddings.embed_query("texto de prueba")
print(f"Mismo texto -> misma salida? {v1 == v2}")
print(f"Similitud: {cosine_similarity(v1, v2):.10f}")
print()

# Rango de valores en el vector
print(f"Min valor en vector: {min(vector):.6f}")
print(f"Max valor en vector: {max(vector):.6f}")
print(f"Promedio: {sum(vector)/len(vector):.6f}")
print("======================================")

## BLOQUE 10 — Mapa conceptual de embeddings en el pipeline RAG

```text

  FASE DE INDEXACION (se hace una vez)
  ====================================
  Documentos originales
       |
       v
  TextSplitter (RecursiveCharacterTextSplitter)
       |  Divide en chunks de ~500 chars
       v
  OpenAIEmbeddings.embed_documents(chunks)
       |  Convierte cada chunk en vector [1536 dims]
       v
  VectorStore (FAISS / Chroma)
       |  Guarda vectores + texto original
       v
  Retriever (interfaz de busqueda)


  FASE DE CONSULTA (pregunta -> respuesta)
  =========================================
  Pregunta del usuario
       |
       v
  OpenAIEmbeddings.embed_query(pregunta)
       |  Convierte la pregunta en vector
       v
  VectorStore.similarity_search(vector, k=3)
       |  Busca los 3 chunks mas cercanos
       v
  Contexto recuperado -> PromptTemplate -> LLM -> Respuesta
```

**¿Por qué embed_query() y embed_documents() separados?**

En producción, indexás documentos una vez y después hacés consultas muchas veces. Tener métodos separados permite optimizar cada operación distinto (por ejemplo, `embed_documents()` puede correr en batch).

## BLOQUE 11 — Comparación final

| Concepto | ¿Qué es? | ¿Para qué sirve? |
|---|---|---|
| **Embedding** | Lista de números que representa significado | Buscar textos similares por cercanía vectorial |
| **`embed_query()`** | Embede UNA consulta | Cuando el usuario pregunta algo |
| **`embed_documents()`** | Embede VARIOS textos | Cuando indexás documentos |
| **Similitud del coseno** | Medida de cercanía entre vectores | Saber qué tan relacionados están dos textos |
| **Dimensiones** | 1536 para ada-002 | Cada dimensión captura un aspecto del significado |
| **Vector store** | Base de datos que guarda vectores | Buscar los K vecinos más cercanos eficientemente |

### ¿Cuándo usar embeddings?

- **Siempre** que hagas RAG (recuperación de documentos)
- Cuando necesites buscar por **significado** y no por palabras exactas
- Para clasificación semántica, clustering de textos, detección de similitud

### Próximos pasos

- **E08 (FAISS)**: guardar estos vectores en una base de datos para búsqueda eficiente
- **E09 (Retriever)**: interfaz estándar para buscar documentos
- **E10 / E11 / E12**: pipeline RAG completo que usa embeddings, vector store y retriever

## BLOQUE 12 — Checks automáticos

In [ ]:
def run_checks():
    assert isinstance(embeddings, OpenAIEmbeddings)
    v = embeddings.embed_query("test")
    assert isinstance(v, list) and len(v) > 100
    v1 = embeddings.embed_query("vacaciones")
    v2 = embeddings.embed_query("vacaciones")
    assert cosine_similarity(v1, v2) > 0.99
    v3 = embeddings.embed_query("futbol")
    assert cosine_similarity(v1, v3) < cosine_similarity(v1, v2)
    print("M3L2 E07 Resolution checks passed")

run_checks()

## Cierre

Hoy aprendiste:

1. **Embedding** = vector numérico que captura el significado de un texto
2. **`OpenAIEmbeddings`** es el wrapper de LangChain para generar embeddings
3. **Similitud del coseno** mide qué tan cercanos son dos vectores
4. Textos similares → vectores cercanos → coseno ~1.0
5. Textos distintos → vectores lejanos → coseno ~0.3-0.5
6. Los embeddings son el fundamento del **sistema RAG** (recuperación por significado)